[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Python from the Start](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)

# Regular Expressions


## What you will be able to do

Find, pull out, and replace text by describing its **shape** rather than its exact characters:
any four digits, a word at the start of a line, a phone number in either format.


## Setup

`re` is the regular expression module. It comes with Python, so there is nothing to install.


In [1]:
import re

print("Ready.")


Ready.


## The idea

Notebook 4 ended where `find` and `replace` stop: they need the exact text. A **regular
expression**, or regex, is a small language for describing a shape instead.

`re.search` looks for the first place a pattern matches.


In [2]:
log = "2026-09-04 ERROR disk full on host web-03"

print(re.search(r"ERROR", log))


<re.Match object; span=(11, 16), match='ERROR'>


That pattern is just literal text, so it behaves like `find`. What comes back is a **match
object**, which carries the matched text and where it was found.

The interesting part starts when the pattern stops being literal. `\d` means "any digit", and
`\d{4}` means "four of them in a row".


In [3]:
print(re.search(r"\d{4}", log))


<re.Match object; span=(0, 4), match='2026'>


Nothing in that pattern mentions `2026`. It describes a shape, and the year is what happened to
fit.


## Worked examples

### The match object, and what happens when there is no match

`.group()` gives the matched text and `.span()` gives its position.


In [4]:
m = re.search(r"\d{4}", log)

print(m.group())
print(m.span())
print(log[11:16])     # the same characters, by slice


2026
(0, 4)
ERROR


When the pattern does not match, `re.search` returns `None`. It does not raise, and it does not
return an empty string.


In [5]:
print(re.search(r"WARN", log))


None


`None` is the single most important thing to remember about `re.search`, because calling
`.group()` on it is the most common regex error there is. Common errors, below, shows it.

Test the result before using it:


In [6]:
m = re.search(r"WARN", log)

if m:
    print("found:", m.group())
else:
    print("no match")


no match


### Always write patterns as raw strings

Notebook 4 introduced `r"..."` and said this notebook would rely on it. Here is why, and it is
not a style preference.

`\b` means "word boundary" in a regex. In an ordinary Python string, `\b` is already a real
escape: it means a **backspace character**.


In [7]:
print(len("\b"), repr("\b"))      # a normal string: one backspace character
print(len(r"\b"), repr(r"\b"))    # a raw string: the two characters you typed


1 '\x08'
2 '\\b'


So the regex never sees what you meant, and the failure is silent.


In [8]:
print(re.findall("\bweb", log))     # backspace, matches nothing
print(re.findall(r"\bweb", log))    # word boundary, matches


[]
['web']


The first returns an empty list with no error and no warning. Write every pattern as a raw
string and this whole class of problem disappears.

### The character classes worth memorizing

These four cover most of what you will write.

| Pattern | Matches |
|---|---|
| `\d` | any digit |
| `\w` | any letter, digit or underscore |
| `\s` | any whitespace |
| `.` | any character at all, except a new line |

Their capitals mean the opposite: `\D` is any non-digit, `\S` any non-whitespace.


In [9]:
print(re.findall(r"\d", log))     # each digit on its own
print(re.findall(r"\w+", log))    # runs of word characters
print(re.findall(r"\S+", log))    # runs of non-whitespace


['2', '0', '2', '6', '0', '9', '0', '4', '0', '3']
['2026', '09', '04', 'ERROR', 'disk', 'full', 'on', 'host', 'web', '03']
['2026-09-04', 'ERROR', 'disk', 'full', 'on', 'host', 'web-03']


`re.findall` returns **every** match as a list, which makes it the fastest way to see what a
pattern is really doing.

### Your own character class

Square brackets mean "any one of these". A dash inside gives a range, and a `^` at the start
flips it to "any one except these".


In [10]:
word = "Philadelphia"

print(re.findall(r"[aeiou]", word))      # any vowel
print(re.findall(r"[a-d]", word))        # a range
print(re.findall(r"[^aeiou]", word))     # anything but a vowel


['i', 'a', 'e', 'i', 'a']
['a', 'd', 'a']
['P', 'h', 'l', 'd', 'l', 'p', 'h']


### How many

A pattern piece on its own matches once. These say how many times to repeat it.

| Pattern | Meaning |
|---|---|
| `+` | one or more |
| `*` | zero or more |
| `?` | zero or one, so optional |
| `{3}` | exactly three |
| `{2,4}` | between two and four |


In [11]:
print(re.findall(r"\d+", log))        # runs of digits
print(re.findall(r"\d{4}", log))      # exactly four
print(re.findall(r"\d{2,4}", log))    # two to four, taking as many as it can


['2026', '09', '04', '03']
['2026']
['2026', '09', '04', '03']


Notice `\d+` gave `2026`, `09`, `04`, `03` while `\d` alone gave ten separate digits. The `+`
is what turns characters into fields.

### Where, not only what

Anchors match a position rather than a character.

| Pattern | Meaning |
|---|---|
| `^` | start of the text |
| `$` | end of the text |
| `\b` | a word boundary |


In [12]:
print(re.findall(r"^\d{4}", log))     # only if the text starts with four digits
print(re.findall(r"\d{2}$", log))     # only at the very end
print(re.findall(r"\bweb\S*", log))   # a word starting with web


['2026']
['03']
['web-03']


### Seeing where a pattern matches

Reading a list of matches tells you what was found but not where. Marking them inside the
original text is easier to check, and it is worth doing whenever a pattern is not behaving.

`re.sub` replaces matches, and `\g<0>` inside the replacement means "whatever was matched".


In [13]:
text = "Call 215-555-0134 or 610-555-9876 before 2026-09-04"

print(re.sub(r"\d{3}-\d{3}-\d{4}", r"[\g<0>]", text))


Call [215-555-0134] or [610-555-9876] before 2026-09-04


The brackets show exactly what the pattern claimed, and that the date at the end was correctly
left alone. Come back to this trick whenever a pattern matches more or less than you expected.


### Groups: pulling out the pieces

Parentheses mark a part of the match you want back separately. This is what turns a search into
an extraction.


In [14]:
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", log)

print(m.group())      # the whole match
print(m.group(1))     # the first parenthesis
print(m.groups())     # all of them


2026-09-04
2026
('2026', '09', '04')


Counting parentheses stops being fun quickly. Name them instead, with `(?P<name>...)`.


In [15]:
m = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", log)

print(m.group("year"), m.group("month"), m.group("day"))
print(m.groupdict())


2026 09 04
{'year': '2026', 'month': '09', 'day': '04'}


`groupdict()` gives a dictionary, which is notebook 9. For now it is enough that names beat
numbers as soon as a pattern has more than one group.

### A gotcha: findall changes shape when you add groups

Without groups, `findall` gives a list of matched strings. **With** groups, it gives the groups
instead, as tuples.


In [16]:
print(re.findall(r"\d{4}-\d{2}", log))        # no groups: the matches
print(re.findall(r"(\d{4})-(\d{2})", log))    # two groups: tuples of the groups


['2026-09']
[('2026', '09')]


This surprises people who added parentheses only for grouping. If you want the parentheses but
not the change in result, `re.finditer` gives match objects instead and leaves the shape alone.


### Replacing and splitting

`re.sub` replaces every match, and `re.split` breaks text wherever a pattern matches.


In [17]:
print(re.sub(r"\d", "#", log))
print(re.split(r"\s+", log))


####-##-## ERROR disk full on host web-##
['2026-09-04', 'ERROR', 'disk', 'full', 'on', 'host', 'web-03']


`re.split(r"\s+", ...)` is the one to reach for with messy spacing, where `.split(" ")` would
leave empty pieces behind.


### Greedy by default

`*` and `+` take as much as they can. That is usually right, and occasionally very wrong.


In [18]:
html = "<b>bold</b> and <i>italic</i>"

print(re.findall(r"<.*>", html))


['<b>bold</b> and <i>italic</i>']


One match, from the first `<` to the **last** `>`, swallowing everything between. `.*` was
greedy and `>` was still satisfiable at the end.

Add `?` after the repeat to make it lazy, taking as little as possible.


In [19]:
print(re.findall(r"<.*?>", html))


['<b>', '</b>', '<i>', '</i>']


Four matches, which is what was wanted. Whenever a pattern returns one enormous match instead
of several small ones, greediness is the reason.


### Putting it together

A pattern with named groups turns unstructured lines into fields.


In [20]:
lines = [
    "2026-09-04 10:22:31 ERROR  disk full on web-03",
    "2026-09-04 10:22:35 INFO   backup finished",
    "2026-09-05 03:14:00 ERROR  timeout on db-01",
]

pattern = r"^(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\S+) (?P<level>\w+)\s+(?P<message>.*)$"

for line in lines:
    m = re.search(pattern, line)
    print(m.group("date"), "|", m.group("level"), "|", m.group("message"))


2026-09-04 | ERROR | disk full on web-03
2026-09-04 | INFO | backup finished
2026-09-05 | ERROR | timeout on db-01


That is the shape of most real text work: describe the line once, then read fields off every
line by name.

One more, because it is the pattern people look up most often.


In [21]:
contacts = "write to ada@example.com or bob.h@sub.example.org"

print(re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", contacts))


['ada@example.com', 'bob.h@sub.example.org']


Read it in pieces: some run of word characters, dots, plus or dash; an `@`; a host; a dot; and
the rest. Note the `\.` in the middle, which matches a literal dot, because a bare `.` would
match any character at all.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/05-regular-expressions-solutions.ipynb).

Use this text for tasks 1 to 4:

```
sample = "Order A-1234 shipped 2026-03-15, order B-99 shipped 2026-04-02"
```

**1.** Print every run of digits in `sample`.


In [22]:
# your code here


**2.** Print every date in `sample`, matching four digits, a dash, two digits, a dash, two
digits.


In [23]:
# your code here


**3.** Print every order code: one capital letter, a dash, then one or more digits.


In [24]:
# your code here


**4.** Use `re.sub` to mark every date in `sample` by wrapping it in square brackets, and print
the result.


In [25]:
# your code here


**5.** Use `re.search` with named groups to pull the year, month and day out of
`"2026-03-15"`, then print them on one line separated by slashes.


In [26]:
# your code here


**6.** `re.search(r"ZZZ", sample)` finds nothing. Write code that prints `not found` instead of
raising an error.


In [27]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### AttributeError: you called `.group()` on nothing

This is the most common regex error, by a wide margin. `re.search` returned `None` because the
pattern did not match, and `None` has no `.group()`.


In [28]:
log = "2026-09-04 ERROR disk full on host web-03"

m = re.search(r"WARN", log)
print(m.group())


AttributeError: 'NoneType' object has no attribute 'group'

`'NoneType' object has no attribute 'group'` always means the same thing: **the pattern did not
match**. The problem is in the pattern or the text, not in the line that failed.

Check first, every time:


In [29]:
m = re.search(r"WARN", log)

print(m.group() if m else "no match")


no match


### Nothing to repeat: a quantifier with nothing in front of it

`+`, `*` and `?` repeat the thing before them, so they cannot come first.


In [30]:
re.search(r"*ERROR", log)


PatternError: nothing to repeat at position 0

`nothing to repeat at position 0` means the `*` had nothing to apply to. To match a literal
star, escape it as `\*`.

The name of this error changed in Python 3.13, from `error` to `PatternError`. Both refer to
the same thing, so an older Python will show `error: nothing to repeat` instead.


### Unbalanced parenthesis

A group has to be closed.


In [31]:
re.search(r"(\d{4}", log)


PatternError: missing ), unterminated subpattern at position 0

`missing ), unterminated subpattern` means an opening parenthesis was never closed. To match a
literal parenthesis rather than open a group, escape it: `\(`.


## Recap

- A regex describes the **shape** of text, which is what `find` and `replace` could not do.
- Write every pattern as a raw string, `r"..."`, because `\b` and friends are string escapes too.
- `\d`, `\w`, `\s` and `.` cover most patterns; `[...]` builds your own class.
- `+ * ? {n}` say how many, and they are **greedy** unless you add `?`.
- `re.search` finds the first match or returns `None`; `re.findall` returns all of them.
- Parentheses capture, `(?P<name>...)` names what they capture, and `findall` returns tuples
  once groups are present.
- `re.sub` with `\g<0>` marks matches in place, which is the fastest way to debug a pattern.


## What is next

**Notebook 6, Booleans and Comparison**, where the `if m:` test used throughout this notebook
gets a proper explanation: what counts as true, what counts as false, and why `is` is not `==`.


---

&#8592; **Previous:** [Strings](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/04-strings.ipynb)  &nbsp;·&nbsp;  [Python from the Start Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)
